In [1]:
import csv
import random
from datetime import datetime, timedelta

# -----------------------
# Configuration
# -----------------------
NUM_MEMBERS = 100
ORG_ID = 55

NUM_DISABLED = 20
NUM_FEMALE = 75
NUM_MALE = 25
NUM_WIDOWED_FEMALE = 3
NUM_CHILDREN = 22
NUM_ORPHAN_CHILDREN = 2

# Age ranges
CHILD_AGE_RANGE = (0, 17)
YOUTH_AGE_RANGE = (18, 30)
ADULT_AGE_RANGE = (31, 60)
ELDERLY_AGE_RANGE = (61, 90)

# Sample first and last names
# -----------------------
# Expanded Zambian first names and last names
# -----------------------

first_names_male = [
    "Brian","Daniel","Felix","Henry","James","Liam","Nathan","Paul","Ryan","Thomas",
    "Victor","Xander","Zane","Benjamin","David","Frank","Jacob","Leo","Nathan","Emmanuel",
    "Patrick","Simon","Stephen","Joseph","Michael","John","Isaac","Caleb","Samuel","Elijah",
    "Lucas","Matthew","Peter","Mark","Jonathan","Ezekiel","Gabriel","Aaron","Davidson","Christopher",
    "George","Leonard","Martin","Phillip","Richard","Samuelson","Victoriano","Wilfred","Austin","Anthony",
    "Benjaminson","Clive","Dominic","Edwin","Frederick","Harrison","Kenneth","Lawrence","Nicholas"
]

first_names_female = [
    "Anna","Chisomo","Esther","Gloria","Irene","Kavita","Maria","Olivia","Quinn","Sophia",
    "Uma","Wendy","Yara","Abigail","Clara","Evelyn","Grace","Isabel","Karen","Martha",
    "Angela","Beatrice","Catherine","Diana","Eliza","Felicity","Faith","Helena","Josephine","Lydia",
    "Mercy","Naomi","Patience","Rachel","Rebecca","Roselyn","Selina","Sophia","Theresa","Victoria",
    "Vivian","Winifred","Yvonne","Zainab","Angela","Bernadette","Carol","Delilah","Eleanor","Frances",
    "Genevieve","Hannah","Janet","Kate","Linda","Monica","Nancy","Olga","Priscilla"
]

last_names = [
    "Phiri","Mwansa","Banda","Zulu","Tembo","Kaponda","Bwalya","Mulenga","Chanda","Mbewe",
    "Chilufya","Ngoma","Sichinga","Mumba","Chilongo","Nkombo","Kabwe","Chansa","Mwila","Lungu",
    "Kalaba","Kamanga","Chileshe","Ngulube","Sikazwe","Mwewa","Chikuni","Mwamba","Katongo","Chikoti",
    "Chinyama","Chikonde","Muntanga","Chilongo","Chola","Mweemba","Nkandu","Phiri-Kabwe","Chikupi","Chakamba",
    "Kaluba","Chisala","Nkoloma","Bwalya-Mulenga","Chama","Kalunga","Mbewe-Chilufya","Nkonde","Chibwe","Zimba",
    "Lukwesa","Chishimba","Chiti","Musonda","Chileshe-Kalaba","Mutale","Lundazi","Chama-Lungu"
]

# -----------------------
# Helper functions
# -----------------------
def random_name(gender):
    first = random.choice(first_names_female if gender=="Female" else first_names_male)
    last = random.choice(last_names)
    return f"{first} {last}"

def random_phone(idx):
    return f"0971{idx:06d}"

def random_email(name, idx):
    return f"{name.lower().replace(' ', '.')}{idx}@example.com"

def random_date_of_birth(age):
    today = datetime.today()
    birth_year = today.year - age
    month = random.randint(1,12)
    day = random.randint(1,28)  # safe
    return f"{birth_year}-{month:02d}-{day:02d}"

def random_join_date():
    start = datetime(2020,1,1)
    end = datetime(2025,12,31)
    delta = (end - start).days
    d = start + timedelta(days=random.randint(0, delta))
    return d.strftime("%Y-%m-%d")

# -----------------------
# Predefine distributions
# -----------------------
members = []

# Gender list
genders = ["Female"]*NUM_FEMALE + ["Male"]*NUM_MALE
random.shuffle(genders)

# Disabled flags
disabled_flags = [True]*NUM_DISABLED + [False]*(NUM_MEMBERS-NUM_DISABLED)
random.shuffle(disabled_flags)

# Children indices
child_indices = random.sample(range(NUM_MEMBERS), NUM_CHILDREN)
orphan_child_indices = random.sample(child_indices, NUM_ORPHAN_CHILDREN)

# Widowed female indices
female_indices = [i for i,g in enumerate(genders) if g=="Female"]
widowed_female_indices = random.sample(female_indices, NUM_WIDOWED_FEMALE)

# -----------------------
# Generate members
# -----------------------
for i in range(NUM_MEMBERS):
    gender = genders[i]
    disabled = disabled_flags[i]

    # Age
    if i in child_indices:
        age = random.randint(*CHILD_AGE_RANGE)
        orphan = i in orphan_child_indices
    else:
        age = random.randint(YOUTH_AGE_RANGE[0], ELDERLY_AGE_RANGE[1])
        orphan = False

    # Widowed only if female
    widowed = i in widowed_female_indices if gender=="Female" else False

    dob = random_date_of_birth(age)
    join = random_join_date()

    # NRC or guardian
    if age >= 18:
        nrc = f"{random.randint(100000,999999)}/{ORG_ID}/{i+1}"
        guardian_name = None
        guardian_phone = None
    else:
        nrc = None
        guardian_name = random_name(random.choice(["Male","Female"]))
        guardian_phone = random_phone(i+1000)

    full_name = random_name(gender)
    email = random_email(full_name, i+1)
    phone = random_phone(i)

    members.append({
        "full_name": full_name,
        "email": email,
        "phone": phone,
        "gender": gender,
        "date_of_birth": dob,
        "date_joined": join,
        "age": age,
        "disabled": disabled,
        "orphan": orphan,
        "widowed": widowed,
        "nrc": nrc,
        "guardian_name": guardian_name,
        "guardian_phone": guardian_phone,
        "status": "Active",
        "organization_id": ORG_ID
    })

# Shuffle final list so distribution looks natural
random.shuffle(members)

# -----------------------
# Write CSV
# -----------------------
with open("members_exact_distribution.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=members[0].keys())
    writer.writeheader()
    writer.writerows(members)

print("✅ CSV 'members_exact_distribution.csv' created with exact distributions.")

✅ CSV 'members_exact_distribution.csv' created with exact distributions.
